**Capítulo 2 – Um Projeto de Machine Learning de Ponta a Ponta**

*Este notebook contém uma adaptação prática dos exemplos de código do Capítulo 2 do livro "Hands-On Machine Learning". Vamos construir um modelo de previsão de preços de imóveis na Califórnia, explorando todas as etapas essenciais: coleta de dados, EDA (análise exploratória), limpeza (imputação, tratamento de categorias), pipelines e avaliação de modelos (Regressão, Árvores, Random Forest).*

# Configuração
*Configurando o ambiente para garantir reprodutibilidade e visualização adequada.*

In [2]:
import sys
assert sys.version_info >= (3, 7)

from packaging import version
import sklearn
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

In [3]:
# Configurações de visualização padrão
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [4]:
import numpy as np

# Definindo uma semente aleatória para gerar sempre os mesmos resultados nas amostras aleatórias
np.random.seed(42)

# Obtendo os Dados

*A primeira etapa funcional de um projeto de Machine Learning é obter os dados em estado bruto e carregá-los em nossa estrutura de análise. Aqui baixamos o conjunto de preços das casas na Califórnia que usaremos durante todo o capítulo.*

## Baixando os dados

In [5]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

/tmp/ipykernel_50628/266164749.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tarball.extractall(path="datasets")


### Uma Rápida Olhada na Estrutura de Dados
*Vamos examinar os dados com os métodos nativos do Pandas para entender o que temos em mãos.*

In [ ]:
housing.head()

In [ ]:
housing.info()

*Observação: O atributo `ocean_proximity` é categórico. Vamos ver quais são as opções e quantos distritos pertencem a cada uma delas usando a função `value_counts()`.*

In [ ]:
housing["ocean_proximity"].value_counts()

In [ ]:
housing.describe()

*Plotar histogramas de cada atributo numérico é uma das melhores formas de "sentir" os dados à primeira vista. Eles revelam distribuições "pesadas na cauda" (tail-heavy) e atributos operando em escalas muito diferentes, que precisaremos ajustar posteriormente com Feature Scaling.*

In [ ]:
# Extra command to tell matplotlib to plot inline when running
%matplotlib inline

housing.hist(bins=50, figsize=(12, 8))
plt.show()

### Criando o Conjunto de Testes (Train/Test Split)

*Precisamos separar os dados antes de explorarmos a fundo, para evitar o "Snooping Bias" (vazamento de dados visuais). Faremos isso de forma estratificada na renda mediana (`median_income`), pois ela é uma métrica crucial para o preço das casas.*

In [ ]:
import numpy as np

# Vamos criar categorias de renda (income_cat) para fazer a estratificação equilibrada
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6.0, np.inf],
                               labels=[1, 2, 3, 4, 5])

housing["income_cat"].value_counts().sort_index().plot.bar(rot=0, grid=True)
plt.xlabel("Categoria de Renda (Income Category)")
plt.ylabel("Número de distritos")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Agora fazemos a separação estratificada garantindo as mesmas proporções de renda no teste e treino
strat_train_set, strat_test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42)

# Podemos remover a coluna temporária "income_cat" e resetar a base
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

# Fazemos uma cópia para preservar o conjunto de treino original
housing = strat_train_set.copy()

# Descobrindo e Visualizando os Dados para Obter Insights

*Nesta etapa, exploramos visualmente os dados para captar padrões, outliers e relações entre variáveis. Essa intuição orienta as próximas decisões de pré-processamento e modelagem.*

## Visualizando Dados Geograficos

*O dataset possui latitude e longitude. Ao plotar esses pontos, conseguimos ver a distribuicao dos distritos e onde estao concentrados os valores mais altos.*

In [ ]:
# Funcao auxiliar para salvar figuras (opcional)
IMAGES_PATH = Path() / "images" / "end_to_end_project"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", grid=True)
save_fig("bad_visualization_plot")
plt.show()

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", grid=True, alpha=0.2)
save_fig("better_visualization_plot")
plt.show()

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", grid=True,
             s=housing["population"] / 100, label="population",
             c="median_house_value", cmap="jet", colorbar=True,
             legend=True, sharex=False, figsize=(10, 7))
save_fig("housing_prices_scatterplot")
plt.show()

*Observacao tecnica: o argumento `sharex=False` evita um bug visual do Pandas que pode esconder os valores no eixo x.*

*A proxima celula gera uma figura mais bonita: ela adiciona um mapa da California ao fundo e melhora a leitura visual.*

In [ ]:
# Baixar a imagem da California para o fundo do grafico
filename = "california.png"
if not (IMAGES_PATH / filename).is_file():
    homl3_root = "https://github.com/ageron/handson-ml3/raw/main/"
    url = homl3_root + "images/end_to_end_project/" + filename
    print("Downloading", filename)
    urllib.request.urlretrieve(url, IMAGES_PATH / filename)

housing_renamed = housing.rename(columns={
    "latitude": "Latitude", "longitude": "Longitude",
    "population": "Population",
    "median_house_value": "Median house value (USD)"})

housing_renamed.plot(
    kind="scatter", x="Longitude", y="Latitude",
    s=housing_renamed["Population"] / 100, label="Population",
    c="Median house value (USD)", cmap="jet", colorbar=True,
    legend=True, sharex=False, figsize=(10, 7))

california_img = plt.imread(IMAGES_PATH / filename)
axis = -124.55, -113.95, 32.45, 42.05
plt.axis(axis)
plt.imshow(california_img, extent=axis)

save_fig("california_housing_prices_plot")
plt.show()

## Procurando Correlacoes

*Uma forma rapida de entender quais variaveis estao mais relacionadas ao preco e usar a matriz de correlacao.*

*Nota: no Pandas 2.0+, o argumento `numeric_only` passou a ser `False` por padrao. Para evitar erro ao calcular correlacoes, definimos como `True`.*

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
from pandas.plotting import scatter_matrix

attributes = ["median_house_value", "median_income", "total_rooms",
              "housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12, 8))
save_fig("scatter_matrix_plot")
plt.show()

In [ ]:
housing.plot(kind="scatter", x="median_income", y="median_house_value",
             alpha=0.1, grid=True)
save_fig("income_vs_house_value_scatterplot")
plt.show()

## Experimentando Combinacoes de Atributos

*Algumas combinacoes de atributos fazem mais sentido do que as colunas brutas. Vamos criar razoes simples e ver se elas se correlacionam mais com o preco.*

In [ ]:
housing["rooms_per_house"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_ratio"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["people_per_house"] = housing["population"] / housing["households"]

corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

# Preparando os Dados para Algoritmos de Machine Learning

*Agora vamos separar o alvo (preco) das features, e iniciar o preparo: limpeza de dados ausentes, tratamento de atributos categoricos e escalonamento.*

Vamos voltar ao conjunto de treino original e separar o alvo (`median_house_value`):

In [ ]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

## Limpeza de Dados (Data Cleaning)

*No livro, existem tres opcoes para lidar com valores ausentes (`NaN`) no atributo `total_bedrooms`.*

```python
housing.dropna(subset=["total_bedrooms"], inplace=True)    # opcao 1
housing.drop("total_bedrooms", axis=1)                    # opcao 2
median = housing["total_bedrooms"].median()               # opcao 3
housing["total_bedrooms"].fillna(median, inplace=True)
```

*Vamos criar copias para nao alterar o dataset original durante a demonstracao.*

In [ ]:
null_rows_idx = housing.isnull().any(axis=1)
housing.loc[null_rows_idx].head()

In [ ]:
housing_option1 = housing.copy()
housing_option1.dropna(subset=["total_bedrooms"], inplace=True)
housing_option1.loc[null_rows_idx].head()

In [ ]:
housing_option2 = housing.copy()
housing_option2.drop("total_bedrooms", axis=1, inplace=True)
housing_option2.loc[null_rows_idx].head()

In [ ]:
housing_option3 = housing.copy()
median = housing["total_bedrooms"].median()
housing_option3["total_bedrooms"].fillna(median, inplace=True)
housing_option3.loc[null_rows_idx].head()

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

housing_num = housing.select_dtypes(include=[np.number])
imputer.fit(housing_num)

X = imputer.transform(housing_num)
housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)
housing_tr.loc[null_rows_idx].head()

## Tratando Atributos Categoricos

*O atributo `ocean_proximity` precisa ser convertido em numeros. Podemos fazer isso via codificacao ordinal ou one-hot.*

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

housing_cat = housing[["ocean_proximity"]]
ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)
housing_cat_encoded[:8]

In [ ]:
ordinal_encoder.categories_

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

*Por padrao, o `OneHotEncoder` gera uma matriz esparsa. Se quiser visualizar como array denso, basta usar `toarray()`.*

In [ ]:
housing_cat_1hot.toarray()

*Alternativamente, voce pode setar `sparse_output=False` na criacao do encoder (o parametro `sparse` foi renomeado para `sparse_output` a partir do Scikit-Learn 1.2).*

In [ ]:
cat_encoder = OneHotEncoder(sparse_output=False)
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

In [ ]:
cat_encoder.categories_

## Feature Scaling

*As escalas das variaveis numericas sao muito diferentes. Por isso aplicamos padronizacao (StandardScaler) ou normalizacao (MinMaxScaler) para facilitar o aprendizado dos modelos.*

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
housing_num_min_max_scaled = min_max_scaler.fit_transform(housing_num)

std_scaler = StandardScaler()
housing_num_std_scaled = std_scaler.fit_transform(housing_num)

In [ ]:
# Visualizando cauda longa (distribuicoes enviesadas)
fig, axs = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
housing["population"].hist(ax=axs[0], bins=50)
housing["population"].apply(np.log).hist(ax=axs[1], bins=50)
axs[0].set_xlabel("Population")
axs[1].set_xlabel("Log of population")
axs[0].set_ylabel("Numero de distritos")
save_fig("long_tail_plot")
plt.show()

*E se substituirmos cada valor pelo seu percentil? Isso gera uma distribuicao quase uniforme e pode ajudar alguns algoritmos.*

In [ ]:
percentiles = [np.percentile(housing["median_income"], p)
               for p in range(1, 100)]
flattened_median_income = pd.cut(housing["median_income"],
                                 bins=[-np.inf] + percentiles + [np.inf],
                                 labels=range(1, 100 + 1))
flattened_median_income.hist(bins=50)
plt.xlabel("Percentil da renda mediana")
plt.ylabel("Numero de distritos")
plt.show()

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

age_simil_35 = rbf_kernel(housing[["housing_median_age"]], [[35]], gamma=0.1)
age_simil_35[:5]

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor

some_new_data = housing[["median_income"]].iloc[:5]

model = TransformedTargetRegressor(LinearRegression(),
                                   transformer=StandardScaler())
model.fit(housing[["median_income"]], housing_labels)
predictions = model.predict(some_new_data)
predictions

## Transformers Customizados

*Podemos criar transformadores simples com `FunctionTransformer` e tambem classes mais completas seguindo a API do Scikit-Learn.*

In [ ]:
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log, inverse_func=np.exp)
log_pop = log_transformer.transform(housing[["population"]])
log_pop[:5]

In [ ]:
rbf_transformer = FunctionTransformer(rbf_kernel,
                                      kw_args=dict(Y=[[35.]], gamma=0.1))
age_simil_35 = rbf_transformer.transform(housing[["housing_median_age"]])
age_simil_35[:5]

In [ ]:
ratio_transformer = FunctionTransformer(lambda X: X[:, [0]] / X[:, [1]])
ratio_transformer.transform(np.array([[1., 2.], [3., 4.]]))

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted

class StandardScalerClone(BaseEstimator, TransformerMixin):
    def __init__(self, with_mean=True):
        self.with_mean = with_mean

    def fit(self, X, y=None):
        X = check_array(X)
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self)
        X = check_array(X)
        if self.with_mean:
            X = X - self.mean_
        return X / self.scale_

In [ ]:
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

## Pipelines de Transformacao

*Agora vamos encadear as etapas de preparo em pipelines reutilizaveis.*

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector

num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"))

preprocessing = ColumnTransformer([
    ("num", num_pipeline, make_column_selector(dtype_include=np.number)),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
])

housing_prepared = preprocessing.fit_transform(housing)

In [ ]:
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

ratio_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(column_ratio, feature_names_out=ratio_name),
    StandardScaler())

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)

default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler())

preprocessing = ColumnTransformer([
        ("bedrooms", ratio_pipeline, ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline, ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline, ["population", "households"]),
        ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                               "households", "median_income"]),
        ("geo", cluster_simil, ["latitude", "longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder=default_num_pipeline)

housing_prepared = preprocessing.fit_transform(housing)
housing_prepared.shape

# Selecionar e Treinar um Modelo

*Com o dataset preparado, vamos treinar alguns modelos e avaliar seu desempenho.*

## Treinando e Avaliando no Conjunto de Treino

In [ ]:
lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)

In [ ]:
housing_predictions = lin_reg.predict(housing)
housing_predictions[:5].round(-2)

In [ ]:
housing_labels.iloc[:5].values

**Aviso:** em versoes recentes do Scikit-Learn, use `root_mean_squared_error()` para RMSE. O bloco abaixo cria essa funcao se ela nao existir.

In [ ]:
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error

    def root_mean_squared_error(labels, predictions):
        return mean_squared_error(labels, predictions, squared=False)

lin_rmse = root_mean_squared_error(housing_labels, housing_predictions)
lin_rmse

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels)

housing_predictions = tree_reg.predict(housing)
tree_rmse = root_mean_squared_error(housing_labels, housing_predictions)
tree_rmse

## Avaliacao Melhor com Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

tree_rmses = -cross_val_score(tree_reg, housing, housing_labels,
                              scoring="neg_root_mean_squared_error", cv=10)
pd.Series(tree_rmses).describe()

In [ ]:
lin_rmses = -cross_val_score(lin_reg, housing, housing_labels,
                              scoring="neg_root_mean_squared_error", cv=10)
pd.Series(lin_rmses).describe()

**Aviso:** a proxima celula pode levar alguns minutos.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing,
                           RandomForestRegressor(random_state=42))
forest_rmses = -cross_val_score(forest_reg, housing, housing_labels,
                                scoring="neg_root_mean_squared_error", cv=10)

pd.Series(forest_rmses).describe()

*Comparando erro de treino e validacao para observar overfitting.*

In [ ]:
forest_reg.fit(housing, housing_labels)
housing_predictions = forest_reg.predict(housing)
forest_rmse = root_mean_squared_error(housing_labels, housing_predictions)
forest_rmse

# Ajuste Fino do Modelo

*Agora ajustamos hiperparametros com Grid Search e Randomized Search.*

## Grid Search

**Aviso:** a proxima celula pode demorar alguns minutos.

In [ ]:
from sklearn.model_selection import GridSearchCV

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])
param_grid = [
    {"preprocessing__geo__n_clusters": [5, 8, 10],
     "random_forest__max_features": [4, 6, 8]},
    {"preprocessing__geo__n_clusters": [10, 15],
     "random_forest__max_features": [6, 8, 10]},
]

grid_search = GridSearchCV(full_pipeline, param_grid, cv=3,
                           scoring="neg_root_mean_squared_error")
grid_search.fit(housing, housing_labels)

*Voce pode ver todos os hiperparametros disponiveis com `full_pipeline.get_params().keys()`.*

In [ ]:
print(str(full_pipeline.get_params().keys())[:1000] + "...")

*Melhor combinacao encontrada:*

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_estimator_

*Pontuacoes por combinacao:*

In [ ]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)

cv_res = cv_res[["param_preprocessing__geo__n_clusters",
                 "param_random_forest__max_features", "split0_test_score",
                 "split1_test_score", "split2_test_score", "mean_test_score"]]
score_cols = ["split0", "split1", "split2", "mean_test_rmse"]
cv_res.columns = ["n_clusters", "max_features"] + score_cols
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)

cv_res.head()

## Randomized Search

**Aviso:** a proxima celula pode demorar alguns minutos.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distribs = {"preprocessing__geo__n_clusters": randint(low=3, high=50),
                  "random_forest__max_features": randint(low=2, high=20)}

rnd_search = RandomizedSearchCV(
    full_pipeline, param_distributions=param_distribs, n_iter=10, cv=3,
    scoring="neg_root_mean_squared_error", random_state=42)

rnd_search.fit(housing, housing_labels)

In [ ]:
cv_res = pd.DataFrame(rnd_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
cv_res = cv_res[["param_preprocessing__geo__n_clusters",
                 "param_random_forest__max_features", "split0_test_score",
                 "split1_test_score", "split2_test_score", "mean_test_score"]]
cv_res.columns = ["n_clusters", "max_features"] + score_cols
cv_res[score_cols] = -cv_res[score_cols].round().astype(np.int64)
cv_res.head()

## Analisando os Melhores Modelos e Seus Erros

In [ ]:
final_model = rnd_search.best_estimator_
feature_importances = final_model["random_forest"].feature_importances_

sorted(zip(feature_importances,
           final_model["preprocessing"].get_feature_names_out()),
           reverse=True)[:10]

## Avaliando o Sistema no Conjunto de Teste

In [ ]:
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

final_predictions = final_model.predict(X_test)

final_rmse = root_mean_squared_error(y_test, final_predictions)
final_rmse

*Podemos calcular um intervalo de confianca para o RMSE usando bootstrap.*

In [ ]:
from scipy import stats

def rmse(squared_errors):
    return np.sqrt(np.mean(squared_errors))

confidence = 0.95
squared_errors = (final_predictions - y_test) ** 2
boot_result = stats.bootstrap([squared_errors], rmse,
                              confidence_level=confidence, random_state=42)
rmse_lower, rmse_upper = boot_result.confidence_interval
rmse_lower, rmse_upper

## Persistencia de Modelo usando joblib

*Agora salvamos o modelo final para reutilizacao futura.*

Salvar o modelo final:

In [ ]:
import joblib

joblib.dump(final_model, "my_california_housing_model.pkl")

Exemplo simples de uso em producao:

In [ ]:
final_model_reloaded = joblib.load("my_california_housing_model.pkl")

new_data = housing.iloc[:5]
predictions = final_model_reloaded.predict(new_data)
predictions

*Voce poderia usar `pickle`, mas o `joblib` costuma ser mais eficiente.*